# 玻璃棒 · 逐算子调试

从上到下逐格执行；每格使用前面的 Dataset，算子调用直接写在 cell 中。这里只拆开现有逻辑，不修复或重写业务结果。

表格展示真实字段，长文本/嵌套内容可滚动查看。`show(ds, columns=[...], n=100)`只控制展示；`show(ds)`看全部列，`ds.take(1)[0]`取完整原行。没有增加展示用的数据字段。

入口是采集的三个原始文件。模型调用有单独 cell，执行该格才调用。先看原始概念记录，再一起往下检查；不要先 Run All。修改代码、配置或输入后换新的 RUN，旧结果保留。


In [ ]:
from pathlib import Path
import sys, json, html
from IPython.display import display, HTML
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from demiflow.standalone import local_data
from curation.v4.contracts import snapshot, immutable, digest, source_code, runtime_version
from curation.v4.pipeline import DEFAULT
from curation.v4.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks,
    ReadDocument, CleanDocument, CheckImage, CountMaterial, NestMaterial,
    merge_concept, distinct, fill_material_counts, model_input)
from curation.v4.ops.prompt_operators import PrepareIdentity, ApplyIdentity
from curation.v4.ops.prompt_config import knowledge_prompt_pack, prompt_execution_options, save_prompt_config
from curation.v4.ops.source_blocks import BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection, merge_block_decisions
from curation.v4.ops.multimodal import SelectAvailableImages, BatchImageSelection, ApplyImageSelection, merge_image_decisions, SelectRelatedMaterials
from curation.v4.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.v4.ops.paragraph_similarity import EmbedParagraphBatch, ParagraphRows
from curation.v4.ops.paragraphs import ApplyParagraphs, ApplyParagraphReview, SelectRetainedParagraphs
from curation.v4.ops.paragraph_pipeline import RouteConceptMaterials, PrepareVerifiedParagraphs, BuildLocalMergeGroups, ApplyLocalIntegration, SourceCatalog, FormatTopicArticle, FinalKnowledgeRecord
from curation.v4.ops.cross_batch import PlanCrossBatchReview, ApplyCrossBatchReview
from curation.v4.ops.paragraph_merge import ApplyParagraphMerge
from curation.v4.ops.topic_quality import PrepareTopicRepairs, ApplyTopicRepairs, RetainReviewedTopics, PrepareTopicVerification
from curation.v4.ops.topic_articles import TopicRows

DATASET = ROOT / 'datasets/demiwtg'
RUN = ROOT / 'state/curation/v4/glass_operator_debug_v3'
CONCEPT = '玻璃棒'
IDS = ['legacy:' + CONCEPT]
GROUP_SIZE = 256
config = {**DEFAULT, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None,
          'max_output_tokens':16384, 'temperature':0, 'timeout_s':900,
          'block_unit_chars':1800, 'block_batch_chars':8000,
          'joint_text_chars':2500, 'image_batch_size':4,
          'text_embedding_model':str(ROOT.parent / 'models/Qwen3-Embedding-0.6B'),
          'image_embedding_model':str(ROOT.parent / 'models/siglip2-base-patch16-224')}
tables = RUN / 'datasets'
knowledge_run = RUN / 'knowledge'

def show(ds, columns=None, n=100):
    from curation.notebook_image_preview import show as preview
    return preview(ds, columns=columns, n=n, run=RUN, dataset=DATASET)


### 冻结本次输入、代码和配置

In [ ]:
concept_source = {'kind':'legacy_concepts', **snapshot(DATASET / 'meta/concepts.json')}
document_source = {'kind':'legacy_docs', **snapshot(DATASET / 'meta/docs.jsonl')}
image_source = {'kind':'legacy_images', **snapshot(DATASET / 'meta/images.jsonl')}
notebook = json.loads((ROOT / 'curation/v4/glass_operator_debug.ipynb').read_text())
manifest = {'sources':[concept_source, document_source, image_source],
            'ids':IDS, 'group_size':GROUP_SIZE, 'config':config,
            'code':source_code(), 'runtime':runtime_version(),
            'cells':[''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code']}
from curation.notebook_image_preview import preserve_display_version
manifest = preserve_display_version(RUN, manifest)
immutable(RUN / 'manifest.json', manifest)
version = digest(manifest)
pack, prompt_text = knowledge_prompt_pack(config)
options = prompt_execution_options(RUN, config)
save_prompt_config(RUN, prompt_text, options)
data = local_data(prompt_packs={'knowledge.yaml':pack}, prompt_options=options)


### 1．read_records · 原始概念记录

In [ ]:
concept_records = data.read_records(DATASET / 'meta/concepts.json', format='json', item_prefix='concepts.item',
    report_path=RUN / 'source_status/concepts.json')
# 这里只筛选要看的行；concept_records仍然是原文件的数据流。
show(concept_records.filter(lambda r: r['error'] is None and isinstance(r['value'],dict) and r['value'].get('name') == CONCEPT), n=1)


### 2．ConceptFromRecord · 转换概念字段

In [ ]:
concepts = concept_records.filter(lambda r: r['error'] is None and isinstance(r['value'],dict)).map(ConceptFromRecord(concept_source))
show(concepts.filter(lambda r:r['concept_ref'] in IDS), n=1)


### 3．SelectConcept · 筛选玻璃棒

In [ ]:
selected_concepts = await concepts.map(SelectConcept(IDS)).filter(lambda r: r['selected']).reduce_by_key('concept_ref', merge_concept).checkpoint_async(tables / 'selected_concepts.jsonl', version=version)
show(selected_concepts, columns=['concept_ref', 'name', 'aliases', 'qid', 'source_records'])


### 4．read_records · 原始文档清单

In [ ]:
document_records = data.read_records(DATASET / 'meta/docs.jsonl', report_path=RUN / 'source_status/documents.json')
show(document_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict) and CONCEPT in r['value'].get('concepts', [])), n=5)


### 5．DocumentFromRecord · 转换文档字段

In [ ]:
documents = document_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict)).map(DocumentFromRecord(document_source))
show(documents.filter(lambda r: any(ref in IDS for ref in r['concept_refs'])), columns=['doc_id','title','url','path','concept_refs'])


### 6．MaterialLinks + join · 关联入选概念的文档

In [ ]:
document_links = await documents.flat_map(MaterialLinks('doc_id')).join(selected_concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').checkpoint_async(tables / 'document_links.jsonl', version=version)
show(document_links)


### 7．join · 保留关联文档

In [ ]:
selected_documents = await documents.join(document_links.select_columns(['doc_id']).reduce_by_key('doc_id', distinct), on='doc_id', how='semi').checkpoint_async(tables / 'selected_documents.jsonl', version=version)
show(selected_documents, columns=['doc_id', 'title', 'url', 'path'])


### 8．ReadDocument · 读取原始页面文本

In [ ]:
raw_documents = await selected_documents.map_cached(ReadDocument(DATASET), cache_dir=RUN / 'cache/read_documents', version=version).checkpoint_async(tables / 'raw_documents.jsonl', version=version)
show(raw_documents, columns=['doc_id', 'title', 'url', 'raw_text', 'read_status', 'read_error'])


### 9．CleanDocument · 清洗页面正文

In [ ]:
processed_documents = await raw_documents.map_cached(CleanDocument(), cache_dir=RUN / 'cache/clean_documents', version=version).checkpoint_async(tables / 'processed_documents.jsonl', version=version)
show(processed_documents, columns=['doc_id', 'title', 'raw_text', 'clean_text', 'clean_blocks', 'clean_status'])


### 10．read_records · 原始图片清单

In [ ]:
image_records = data.read_records(DATASET / 'meta/images.jsonl', report_path=RUN / 'source_status/images.json')
show(image_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict) and CONCEPT in (r['value'].get('concepts') or r['value'].get('instances') or [])), n=5)


### 11．ImageFromRecord · 转换图片字段

In [ ]:
images = image_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict)).map(ImageFromRecord(image_source))
show(images.filter(lambda r:any(ref in IDS for ref in r['concept_refs'])), columns=['image_id','path','caption','concept_refs'], n=5)


### 12．MaterialLinks + join · 关联入选概念的图片

In [ ]:
image_links = await images.flat_map(MaterialLinks('image_id')).join(selected_concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').checkpoint_async(tables / 'image_links.jsonl', version=version)
show(image_links)


### 13．join · 保留关联图片

In [ ]:
selected_images = await images.join(image_links.select_columns(['image_id']).reduce_by_key('image_id', distinct), on='image_id', how='semi').checkpoint_async(tables / 'selected_images.jsonl', version=version)
show(selected_images, columns=['image_id', 'path', 'caption', 'content_url', 'landing_url'])


### 14．CheckImage · 检查实际图片文件

In [ ]:
processed_images = await selected_images.map_cached(CheckImage(DATASET), cache_dir=RUN / 'cache/check_images', version=version).checkpoint_async(tables / 'processed_images.jsonl', version=version)
show(processed_images, columns=['image_id', 'path', 'byte_status', 'byte_details'])


### 15．CountMaterial · 文档数量和读取状态

In [ ]:
document_counts = await document_links.join(processed_documents.select_columns(['doc_id', 'read_status']), on='doc_id').reduce_by_key('concept_ref', CountMaterial('document_count', 'read_status', 'readable_documents')).checkpoint_async(tables / 'document_counts.jsonl', version=version)
show(document_counts)


### 16．CountMaterial · 图片数量和文件状态

In [ ]:
image_counts = await image_links.join(processed_images.select_columns(['image_id', 'byte_status']), on='image_id').reduce_by_key('concept_ref', CountMaterial('image_count', 'byte_status', 'verified_images')).checkpoint_async(tables / 'image_counts.jsonl', version=version)
show(image_counts)


### 17．join · 将资料数量关联到概念

In [ ]:
concepts_ready = await selected_concepts.join(document_counts, on='concept_ref', how='left').join(image_counts, on='concept_ref', how='left').map(fill_material_counts).checkpoint_async(tables / 'concepts_ready.jsonl', version=version)
show(concepts_ready, columns=['concept_ref', 'name', 'document_count', 'readable_documents', 'image_count', 'verified_images'])


### 18．NestMaterial · 文档按概念关联

In [ ]:
concept_documents = await document_links.join(processed_documents.map(NestMaterial('doc_id', 'documents')), on='doc_id').checkpoint_async(tables / 'concept_documents.jsonl', version=version)
show(concept_documents)


### 19．NestMaterial · 图片按概念关联

In [ ]:
concept_images = await image_links.join(processed_images.map(NestMaterial('image_id', 'images')), on='image_id').checkpoint_async(tables / 'concept_images.jsonl', version=version)
show(concept_images)


### 20．group_batches · 按概念汇集资料

In [ ]:
material_batches = await concept_documents.union(concept_images).group_batches('concept_ref', max_rows=GROUP_SIZE, output='materials').checkpoint_async(tables / 'material_batches.jsonl', version=version)
show(material_batches)


### 21．join · 概念与资料批次

In [ ]:
batches = await concepts_ready.join(material_batches, on='concept_ref', how='left').checkpoint_async(tables / 'batches.jsonl', version=version)
show(batches, columns=['concept_ref', 'name', 'materials'])


### 22．model_input · 转成现有模型算子的输入

In [ ]:
model_inputs = await batches.map(model_input).checkpoint_async(tables / 'model_inputs.jsonl', version=version)
show(model_inputs)


### 23．PrepareIdentity · 准备身份判断输入

In [ ]:
identity_inputs = await model_inputs.map_cached(PrepareIdentity(knowledge_run, config), cache_dir=RUN / 'cache/identity_prepare', version=version).checkpoint_async(tables / 'identity_inputs.jsonl', version=version)
show(identity_inputs, columns=['case_id', 'identity_prompt'])


### 24．identity · 模型调用

In [ ]:
identity_responses = await identity_inputs.map_prompt_async('identity', config='knowledge.yaml', inputs={'payload': 'identity_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1, when=lambda r: not r.get('blocked') and 'identity_prompt' in r).checkpoint_async(tables / 'identity_responses.jsonl', version=version)
show(identity_responses, columns=['prompt_result', 'prompt_error'])


### 25．ApplyIdentity · 解析身份判断

In [ ]:
identified = await identity_responses.map_cached(ApplyIdentity(knowledge_run, config), cache_dir=RUN / 'cache/identity_apply', version=version).checkpoint_async(tables / 'identified.jsonl', version=version)
show(identified, columns=['case_id', 'identity', 'identity_materials', 'identity_unexamined'])


### 26．BuildSourceBlocks · 按原文章节拆正文块

In [ ]:
source_blocks = await identified.map(BuildSourceBlocks(config['block_unit_chars'], body_only=True)).checkpoint_async(tables / 'source_blocks.jsonl', version=version)
show(source_blocks, columns=['case_id', 'source_units'])


### 27．SelectAvailableImages · 收集文件可用的图片

In [ ]:
blocks = await source_blocks.map(SelectAvailableImages()).checkpoint_async(tables / 'blocks.jsonl', version=version)
show(blocks, columns=['case_id', 'available_images', 'image_material_scope'])


### 28．BatchSourceBlocks · 组装文字筛选请求

In [ ]:
text_requests = await blocks.flat_map(BatchSourceBlocks(config['block_batch_chars'])).checkpoint_async(tables / 'text_requests.jsonl', version=version)
show(text_requests, columns=['case_id', 'block_prompt'])


### 29．select_blocks · 模型调用

In [ ]:
text_responses = await text_requests.map_prompt_async('select_blocks', config='knowledge.yaml', inputs={'payload': 'block_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'text_responses.jsonl', version=version)
show(text_responses, columns=['prompt_result', 'prompt_error'])


### 30．ApplyBlockSelection · 解析正文相关性判断

In [ ]:
text_decisions = await text_responses.map_cached(ApplyBlockSelection(relevance_only=True), cache_dir=RUN / 'cache/text_selection', version=version).checkpoint_async(tables / 'text_decisions.jsonl', version=version)
show(text_decisions)


### 31．reduce_by_key · 汇集正文筛选结果

In [ ]:
text_by_concept = await text_decisions.reduce_by_key('case_id', merge_block_decisions).checkpoint_async(tables / 'text_by_concept.jsonl', version=version)
show(text_by_concept)


### 32．BatchImageSelection · 组装图片筛选请求

In [ ]:
image_requests = await blocks.flat_map(BatchImageSelection(config['image_batch_size'])).checkpoint_async(tables / 'image_requests.jsonl', version=version)
show(image_requests, columns=['case_id', 'image_prompt', 'pixel_images'])


### 33．select_images · 模型调用

In [ ]:
image_responses = await image_requests.map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload': 'image_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'image_responses.jsonl', version=version)
show(image_responses, columns=['prompt_result', 'prompt_error'])


### 34．ApplyImageSelection · 解析图片筛选结果

In [ ]:
image_decisions = await image_responses.map_cached(ApplyImageSelection(), cache_dir=RUN / 'cache/image_selection', version=version).checkpoint_async(tables / 'image_decisions.jsonl', version=version)
show(image_decisions)


### 35．reduce_by_key · 汇集图片筛选结果

In [ ]:
image_by_concept = await image_decisions.reduce_by_key('case_id', merge_image_decisions).checkpoint_async(tables / 'image_by_concept.jsonl', version=version)
show(image_by_concept)


### 36．SelectRelatedMaterials · 保留相关原文和图片

In [ ]:
related = await blocks.join(text_by_concept, on='case_id', how='left').join(image_by_concept, on='case_id', how='left').map(SelectRelatedMaterials()).checkpoint_async(tables / 'related.jsonl', version=version)
show(related, columns=['case_id', 'material_pack'])


### 37．PrepareRoutingMaterials · 原生图文关联

In [ ]:
routing_materials = await related.map(PrepareRoutingMaterials()).checkpoint_async(tables / 'routing_materials.jsonl', version=version)
show(routing_materials, columns=['concept', 'passages', 'images', 'native_links', 'unmatched_native_references'])


### 38．RawPassageRows · 展开正文块

In [ ]:
passage_rows = await routing_materials.flat_map(RawPassageRows()).checkpoint_async(tables / 'passage_rows.jsonl', version=version)
show(passage_rows)


### 39．EmbedParagraphBatch · 正文向量

In [ ]:
text_embeddings = await passage_rows.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/input_embeddings', version=version).checkpoint_async(tables / 'text_embeddings.jsonl', version=version)
show(text_embeddings)


### 40．reduce_by_key · 按概念汇集正文向量

In [ ]:
text_vectors = await text_embeddings.flat_map(lambda r: r['items']).reduce_by_key('case_id', lambda a, r: {'case_id': r['case_id'], 'passage_embeddings': {**a['passage_embeddings'], r['source_id']: r}}, initial={'passage_embeddings': {}}).checkpoint_async(tables / 'text_vectors.jsonl', version=version)
show(text_vectors)


### 41．EncodeImageTextMaterials · 图文相似度所需向量

In [ ]:
image_text_vectors = await routing_materials.map_cached(EncodeImageTextMaterials(config['image_embedding_model']), cache_dir=RUN / 'cache/input_image_embeddings', version=version).checkpoint_async(tables / 'image_text_vectors.jsonl', version=version)
show(image_text_vectors)


### 42．RouteConceptMaterials · 分组并关联图片

In [ ]:
routed = await routing_materials.join(text_vectors, on='case_id', how='left').join(image_text_vectors.select_columns(['case_id', 'text_windows', 'image_vectors']), on='case_id', how='left').map(RouteConceptMaterials(config['joint_text_chars'], config['image_batch_size'])).checkpoint_async(tables / 'routed.jsonl', version=version)
show(routed, columns=['concept', 'requests', 'edges', 'overflow_edges'])


### 43．BuildRoutedJointRequest · 联合提炼的实际输入

In [ ]:
joint_requests = await routed.flat_map(lambda r: r['requests']).map(BuildRoutedJointRequest()).checkpoint_async(tables / 'joint_requests.jsonl', version=version)
show(joint_requests, columns=['batch_id', 'joint_prompt', 'pixel_images'])


### 44．joint_paragraphs · 模型调用

In [ ]:
joint_responses = await joint_requests.map_prompt_async('joint_paragraphs', config='knowledge.yaml', inputs={'payload': 'joint_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'joint_responses.jsonl', version=version)
show(joint_responses, columns=['prompt_result', 'prompt_error'])


### 45．ApplyParagraphs · 解析段落和引用

In [ ]:
extracted = await joint_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/paragraph_extract', version=version).checkpoint_async(tables / 'extracted.jsonl', version=version)
show(extracted, columns=['batch_id', 'topics', 'coverage_note'])


### 46．verify_paragraphs · 模型调用

In [ ]:
verify_responses = await extracted.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'verify_responses.jsonl', version=version)
show(verify_responses, columns=['prompt_result', 'prompt_error'])


### 47．ApplyParagraphReview · 解析核验结果

In [ ]:
verified = await verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/paragraph_verify', version=version).checkpoint_async(tables / 'verified.jsonl', version=version)
show(verified, columns=['batch_id', 'topics'])


### 48．join · 将实际像素接回段落

In [ ]:
originals = await verified.join(joint_requests.select_columns(['batch_id', 'pixel_images']), on='batch_id', how='left').map(PrepareVerifiedParagraphs(RUN)).checkpoint_async(tables / 'originals.jsonl', version=version)
show(originals, columns=['batch_id', 'topics'])


## 主题修复与局部整合

下面同样拆开现有实验逻辑。已知修复可能重生重复内容；这里保留真实行为供调试。两个整合轮次分别展开，不在函数或循环中自动跑完。


### initial_repair · PrepareTopicRepairs

In [ ]:
initial_repair_requests = await originals.flat_map(PrepareTopicRepairs()).checkpoint_async(tables / 'initial_repair_requests.jsonl', version=version)
show(initial_repair_requests, columns=['batch_id', 'repair_payload', 'pixel_images'])


### initial_repair · repair_topics · 模型调用

In [ ]:
initial_repair_responses = await initial_repair_requests.map_prompt_async('repair_topics', config='knowledge.yaml', inputs={'payload': 'repair_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'initial_repair_responses.jsonl', version=version)
show(initial_repair_responses, columns=['prompt_result', 'prompt_error'])


### initial_repair · ApplyParagraphs

In [ ]:
initial_repair_extracted = await initial_repair_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/initial_repair_extract', version=version).checkpoint_async(tables / 'initial_repair_extracted.jsonl', version=version)
show(initial_repair_extracted, columns=['batch_id', 'topics'])


### initial_repair · PrepareTopicVerification

In [ ]:
initial_repair_verify_inputs = await initial_repair_extracted.map(PrepareTopicVerification()).checkpoint_async(tables / 'initial_repair_verify_inputs.jsonl', version=version)
show(initial_repair_verify_inputs, columns=['batch_id', 'verify_payload', 'pixel_images'])


### initial_repair · verify_paragraphs · 模型调用

In [ ]:
initial_repair_verify_responses = await initial_repair_verify_inputs.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'initial_repair_verify_responses.jsonl', version=version)
show(initial_repair_verify_responses, columns=['prompt_result', 'prompt_error'])


### initial_repair · ApplyParagraphReview

In [ ]:
initial_repair_checked = await initial_repair_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/initial_repair_verify', version=version).checkpoint_async(tables / 'initial_repair_checked.jsonl', version=version)
show(initial_repair_checked, columns=['batch_id', 'topics'])


### initial_repair · join / reduce_by_key

In [ ]:
initial_repair_with_pixels = initial_repair_checked.join(initial_repair_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left')
initial_repair_results = initial_repair_with_pixels.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repairs':a['repairs']+[r]},initial={'repairs':[]})
initial_repair_planned = initial_repair_requests.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repair_requests':a['repair_requests']+[{'parent_topic_index':r['parent_topic_index']}]},initial={'repair_requests':[]})
show(initial_repair_planned)


### initial_repair · ApplyTopicRepairs

In [ ]:
initial_repair_assembled = await originals.join(initial_repair_results, on='batch_id', how='left').join(initial_repair_planned, on='batch_id', how='left').map(ApplyTopicRepairs()).checkpoint_async(tables / 'initial_repair_assembled.jsonl', version=version)
show(initial_repair_assembled)


### initial_repair · RetainReviewedTopics

In [ ]:
initial_repair_topics = await initial_repair_assembled.flat_map(lambda r: r['rows']).map(RetainReviewedTopics()).checkpoint_async(tables / 'initial_repair_topics.jsonl', version=version)
show(initial_repair_topics, columns=['batch_id', 'topics'])


## 第 1 轮局部整合

### round_1 · SelectRetainedParagraphs

In [ ]:
round_1_retained = await initial_repair_topics.map(SelectRetainedParagraphs()).checkpoint_async(tables / 'round_1_retained.jsonl', version=version)
show(round_1_retained, columns=['content'])


### round_1 · ParagraphRows

In [ ]:
round_1_paragraphs = await round_1_retained.map(lambda r: r['content']).flat_map(ParagraphRows()).checkpoint_async(tables / 'round_1_paragraphs.jsonl', version=version)
show(round_1_paragraphs)


### round_1 · EmbedParagraphBatch

In [ ]:
round_1_embeddings = await round_1_paragraphs.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/round_1_embeddings', version=version).checkpoint_async(tables / 'round_1_embeddings.jsonl', version=version)
show(round_1_embeddings)


### round_1 · reduce_by_key

In [ ]:
round_1_groups = await round_1_embeddings.flat_map(lambda r: r['items']).reduce_by_key('concept', lambda a, r: {'concept': r['concept'], 'items': a['items'] + [r]}, initial={'items': []}).checkpoint_async(tables / 'round_1_groups.jsonl', version=version)
show(round_1_groups)


### round_1 · PlanCrossBatchReview

In [ ]:
round_1_plans = await round_1_groups.map(PlanCrossBatchReview(include_same_batch=True)).checkpoint_async(tables / 'round_1_plans.jsonl', version=version)
show(round_1_plans)


### round_1 · 展开待比较段落对

In [ ]:
round_1_pairs = await round_1_plans.flat_map(lambda r: r['review_requests']).map(lambda r: {**r, 'review_payload': {'items': r['items'], 'reasons': r['reasons']}}).checkpoint_async(tables / 'round_1_pairs.jsonl', version=version)
show(round_1_pairs, columns=['paragraph_ids', 'review_payload'])


### round_1 · review_cross_batch · 模型调用

In [ ]:
round_1_responses = await round_1_pairs.map_prompt_async('review_cross_batch', config='knowledge.yaml', inputs={'payload': 'review_payload'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_responses.jsonl', version=version)
show(round_1_responses, columns=['prompt_result', 'prompt_error'])


### round_1 · ApplyCrossBatchReview

In [ ]:
round_1_reviews = await round_1_responses.map_cached(ApplyCrossBatchReview(), cache_dir=RUN / 'cache/round_1_relations', version=version).checkpoint_async(tables / 'round_1_reviews.jsonl', version=version)
show(round_1_reviews, columns=['paragraph_ids', 'relationship_review', 'next_action'])


### round_1 · 按概念关联原段落和关系判断

In [ ]:
round_1_sources = initial_repair_topics.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'source_rows':a['source_rows']+[r]},initial={'source_rows':[]})
round_1_relations = round_1_reviews.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'review_rows':a['review_rows']+[r]},initial={'review_rows':[]})
round_1_inputs = round_1_groups.join(round_1_relations,on='concept',how='left').join(round_1_sources,on='concept',how='left')
show(round_1_inputs, columns=['concept','review_rows'])


### round_1 · BuildLocalMergeGroups

In [ ]:
round_1_merge_plan = await round_1_inputs.map(BuildLocalMergeGroups()).checkpoint_async(tables / 'round_1_merge_plan.jsonl', version=version)
show(round_1_merge_plan)


### round_1 · 展开局部整合请求

In [ ]:
round_1_merge_requests = await round_1_merge_plan.flat_map(lambda r: r['requests']).checkpoint_async(tables / 'round_1_merge_requests.jsonl', version=version)
show(round_1_merge_requests, columns=['batch_id', 'merge_payload', 'pixel_images'])


### round_1 · merge_paragraphs · 模型调用

In [ ]:
round_1_merge_responses = await round_1_merge_requests.map_prompt_async('merge_paragraphs', config='knowledge.yaml', inputs={'payload': 'merge_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_merge_responses.jsonl', version=version)
show(round_1_merge_responses, columns=['prompt_result', 'prompt_error'])


### round_1 · ApplyParagraphMerge

In [ ]:
round_1_merge_extracted = await round_1_merge_responses.map_cached(ApplyParagraphMerge(), cache_dir=RUN / 'cache/round_1_merge', version=version).checkpoint_async(tables / 'round_1_merge_extracted.jsonl', version=version)
show(round_1_merge_extracted, columns=['batch_id', 'topics', 'merge_validation_issues'])


### round_1 · verify_merged_paragraphs · 模型调用

In [ ]:
round_1_merge_verify_responses = await round_1_merge_extracted.map_prompt_async('verify_merged_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_merge_verify_responses.jsonl', version=version)
show(round_1_merge_verify_responses, columns=['prompt_result', 'prompt_error'])


### round_1 · ApplyParagraphReview

In [ ]:
round_1_merge_checked = await round_1_merge_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/round_1_merge_verify', version=version).checkpoint_async(tables / 'round_1_merge_checked.jsonl', version=version)
show(round_1_merge_checked, columns=['batch_id', 'topics'])


### round_1 · 接回像素并关联原段落

In [ ]:
round_1_merged = round_1_merge_checked.join(round_1_merge_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left').map(PrepareVerifiedParagraphs(RUN))
round_1_merged_by_concept = round_1_merged.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'local_results':a['local_results']+[r]},initial={'local_results':[]})
round_1_assembly_input = round_1_sources.join(round_1_merged_by_concept,on='concept',how='left')
show(round_1_merged, columns=['concept','batch_id','topics'])


### round_1 · ApplyLocalIntegration

In [ ]:
round_1_assembled = await round_1_assembly_input.map(ApplyLocalIntegration()).checkpoint_async(tables / 'round_1_assembled.jsonl', version=version)
show(round_1_assembled)


### round_1 · 展开整合后各批结果

In [ ]:
round_1_rows = await round_1_assembled.flat_map(lambda r: r['rows']).checkpoint_async(tables / 'round_1_rows.jsonl', version=version)
show(round_1_rows, columns=['batch_id', 'topics'])


### round_1_repair · PrepareTopicRepairs

In [ ]:
round_1_repair_requests = await round_1_rows.flat_map(PrepareTopicRepairs()).checkpoint_async(tables / 'round_1_repair_requests.jsonl', version=version)
show(round_1_repair_requests, columns=['batch_id', 'repair_payload', 'pixel_images'])


### round_1_repair · repair_topics · 模型调用

In [ ]:
round_1_repair_responses = await round_1_repair_requests.map_prompt_async('repair_topics', config='knowledge.yaml', inputs={'payload': 'repair_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_repair_responses.jsonl', version=version)
show(round_1_repair_responses, columns=['prompt_result', 'prompt_error'])


### round_1_repair · ApplyParagraphs

In [ ]:
round_1_repair_extracted = await round_1_repair_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/round_1_repair_extract', version=version).checkpoint_async(tables / 'round_1_repair_extracted.jsonl', version=version)
show(round_1_repair_extracted, columns=['batch_id', 'topics'])


### round_1_repair · PrepareTopicVerification

In [ ]:
round_1_repair_verify_inputs = await round_1_repair_extracted.map(PrepareTopicVerification()).checkpoint_async(tables / 'round_1_repair_verify_inputs.jsonl', version=version)
show(round_1_repair_verify_inputs, columns=['batch_id', 'verify_payload', 'pixel_images'])


### round_1_repair · verify_paragraphs · 模型调用

In [ ]:
round_1_repair_verify_responses = await round_1_repair_verify_inputs.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_repair_verify_responses.jsonl', version=version)
show(round_1_repair_verify_responses, columns=['prompt_result', 'prompt_error'])


### round_1_repair · ApplyParagraphReview

In [ ]:
round_1_repair_checked = await round_1_repair_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/round_1_repair_verify', version=version).checkpoint_async(tables / 'round_1_repair_checked.jsonl', version=version)
show(round_1_repair_checked, columns=['batch_id', 'topics'])


### round_1_repair · join / reduce_by_key

In [ ]:
round_1_repair_with_pixels = round_1_repair_checked.join(round_1_repair_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left')
round_1_repair_results = round_1_repair_with_pixels.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repairs':a['repairs']+[r]},initial={'repairs':[]})
round_1_repair_planned = round_1_repair_requests.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repair_requests':a['repair_requests']+[{'parent_topic_index':r['parent_topic_index']}]},initial={'repair_requests':[]})
show(round_1_repair_planned)


### round_1_repair · ApplyTopicRepairs

In [ ]:
round_1_repair_assembled = await round_1_rows.join(round_1_repair_results, on='batch_id', how='left').join(round_1_repair_planned, on='batch_id', how='left').map(ApplyTopicRepairs()).checkpoint_async(tables / 'round_1_repair_assembled.jsonl', version=version)
show(round_1_repair_assembled)


### round_1_repair · RetainReviewedTopics

In [ ]:
round_1_repair_topics = await round_1_repair_assembled.flat_map(lambda r: r['rows']).map(RetainReviewedTopics()).checkpoint_async(tables / 'round_1_repair_topics.jsonl', version=version)
show(round_1_repair_topics, columns=['batch_id', 'topics'])


## 第 2 轮局部整合

### round_2 · SelectRetainedParagraphs

In [ ]:
round_2_retained = await round_1_repair_topics.map(SelectRetainedParagraphs()).checkpoint_async(tables / 'round_2_retained.jsonl', version=version)
show(round_2_retained, columns=['content'])


### round_2 · ParagraphRows

In [ ]:
round_2_paragraphs = await round_2_retained.map(lambda r: r['content']).flat_map(ParagraphRows()).checkpoint_async(tables / 'round_2_paragraphs.jsonl', version=version)
show(round_2_paragraphs)


### round_2 · EmbedParagraphBatch

In [ ]:
round_2_embeddings = await round_2_paragraphs.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/round_2_embeddings', version=version).checkpoint_async(tables / 'round_2_embeddings.jsonl', version=version)
show(round_2_embeddings)


### round_2 · reduce_by_key

In [ ]:
round_2_groups = await round_2_embeddings.flat_map(lambda r: r['items']).reduce_by_key('concept', lambda a, r: {'concept': r['concept'], 'items': a['items'] + [r]}, initial={'items': []}).checkpoint_async(tables / 'round_2_groups.jsonl', version=version)
show(round_2_groups)


### round_2 · PlanCrossBatchReview

In [ ]:
round_2_plans = await round_2_groups.map(PlanCrossBatchReview(include_same_batch=True)).checkpoint_async(tables / 'round_2_plans.jsonl', version=version)
show(round_2_plans)


### round_2 · 展开待比较段落对

In [ ]:
round_2_pairs = await round_2_plans.flat_map(lambda r: r['review_requests']).map(lambda r: {**r, 'review_payload': {'items': r['items'], 'reasons': r['reasons']}}).checkpoint_async(tables / 'round_2_pairs.jsonl', version=version)
show(round_2_pairs, columns=['paragraph_ids', 'review_payload'])


### round_2 · review_cross_batch · 模型调用

In [ ]:
round_2_responses = await round_2_pairs.map_prompt_async('review_cross_batch', config='knowledge.yaml', inputs={'payload': 'review_payload'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_2_responses.jsonl', version=version)
show(round_2_responses, columns=['prompt_result', 'prompt_error'])


### round_2 · ApplyCrossBatchReview

In [ ]:
round_2_reviews = await round_2_responses.map_cached(ApplyCrossBatchReview(), cache_dir=RUN / 'cache/round_2_relations', version=version).checkpoint_async(tables / 'round_2_reviews.jsonl', version=version)
show(round_2_reviews, columns=['paragraph_ids', 'relationship_review', 'next_action'])


### round_2 · 按概念关联原段落和关系判断

In [ ]:
round_2_sources = round_1_repair_topics.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'source_rows':a['source_rows']+[r]},initial={'source_rows':[]})
round_2_relations = round_2_reviews.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'review_rows':a['review_rows']+[r]},initial={'review_rows':[]})
round_2_inputs = round_2_groups.join(round_2_relations,on='concept',how='left').join(round_2_sources,on='concept',how='left')
show(round_2_inputs, columns=['concept','review_rows'])


### round_2 · BuildLocalMergeGroups

In [ ]:
round_2_merge_plan = await round_2_inputs.map(BuildLocalMergeGroups()).checkpoint_async(tables / 'round_2_merge_plan.jsonl', version=version)
show(round_2_merge_plan)


### round_2 · 展开局部整合请求

In [ ]:
round_2_merge_requests = await round_2_merge_plan.flat_map(lambda r: r['requests']).checkpoint_async(tables / 'round_2_merge_requests.jsonl', version=version)
show(round_2_merge_requests, columns=['batch_id', 'merge_payload', 'pixel_images'])


### round_2 · merge_paragraphs · 模型调用

In [ ]:
round_2_merge_responses = await round_2_merge_requests.map_prompt_async('merge_paragraphs', config='knowledge.yaml', inputs={'payload': 'merge_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_2_merge_responses.jsonl', version=version)
show(round_2_merge_responses, columns=['prompt_result', 'prompt_error'])


### round_2 · ApplyParagraphMerge

In [ ]:
round_2_merge_extracted = await round_2_merge_responses.map_cached(ApplyParagraphMerge(), cache_dir=RUN / 'cache/round_2_merge', version=version).checkpoint_async(tables / 'round_2_merge_extracted.jsonl', version=version)
show(round_2_merge_extracted, columns=['batch_id', 'topics', 'merge_validation_issues'])


### round_2 · verify_merged_paragraphs · 模型调用

In [ ]:
round_2_merge_verify_responses = await round_2_merge_extracted.map_prompt_async('verify_merged_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_2_merge_verify_responses.jsonl', version=version)
show(round_2_merge_verify_responses, columns=['prompt_result', 'prompt_error'])


### round_2 · ApplyParagraphReview

In [ ]:
round_2_merge_checked = await round_2_merge_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/round_2_merge_verify', version=version).checkpoint_async(tables / 'round_2_merge_checked.jsonl', version=version)
show(round_2_merge_checked, columns=['batch_id', 'topics'])


### round_2 · 接回像素并关联原段落

In [ ]:
round_2_merged = round_2_merge_checked.join(round_2_merge_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left').map(PrepareVerifiedParagraphs(RUN))
round_2_merged_by_concept = round_2_merged.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'local_results':a['local_results']+[r]},initial={'local_results':[]})
round_2_assembly_input = round_2_sources.join(round_2_merged_by_concept,on='concept',how='left')
show(round_2_merged, columns=['concept','batch_id','topics'])


### round_2 · ApplyLocalIntegration

In [ ]:
round_2_assembled = await round_2_assembly_input.map(ApplyLocalIntegration()).checkpoint_async(tables / 'round_2_assembled.jsonl', version=version)
show(round_2_assembled)


### round_2 · 展开整合后各批结果

In [ ]:
round_2_rows = await round_2_assembled.flat_map(lambda r: r['rows']).checkpoint_async(tables / 'round_2_rows.jsonl', version=version)
show(round_2_rows, columns=['batch_id', 'topics'])


### round_2_repair · PrepareTopicRepairs

In [ ]:
round_2_repair_requests = await round_2_rows.flat_map(PrepareTopicRepairs()).checkpoint_async(tables / 'round_2_repair_requests.jsonl', version=version)
show(round_2_repair_requests, columns=['batch_id', 'repair_payload', 'pixel_images'])


### round_2_repair · repair_topics · 模型调用

In [ ]:
round_2_repair_responses = await round_2_repair_requests.map_prompt_async('repair_topics', config='knowledge.yaml', inputs={'payload': 'repair_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_2_repair_responses.jsonl', version=version)
show(round_2_repair_responses, columns=['prompt_result', 'prompt_error'])


### round_2_repair · ApplyParagraphs

In [ ]:
round_2_repair_extracted = await round_2_repair_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/round_2_repair_extract', version=version).checkpoint_async(tables / 'round_2_repair_extracted.jsonl', version=version)
show(round_2_repair_extracted, columns=['batch_id', 'topics'])


### round_2_repair · PrepareTopicVerification

In [ ]:
round_2_repair_verify_inputs = await round_2_repair_extracted.map(PrepareTopicVerification()).checkpoint_async(tables / 'round_2_repair_verify_inputs.jsonl', version=version)
show(round_2_repair_verify_inputs, columns=['batch_id', 'verify_payload', 'pixel_images'])


### round_2_repair · verify_paragraphs · 模型调用

In [ ]:
round_2_repair_verify_responses = await round_2_repair_verify_inputs.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_2_repair_verify_responses.jsonl', version=version)
show(round_2_repair_verify_responses, columns=['prompt_result', 'prompt_error'])


### round_2_repair · ApplyParagraphReview

In [ ]:
round_2_repair_checked = await round_2_repair_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/round_2_repair_verify', version=version).checkpoint_async(tables / 'round_2_repair_checked.jsonl', version=version)
show(round_2_repair_checked, columns=['batch_id', 'topics'])


### round_2_repair · join / reduce_by_key

In [ ]:
round_2_repair_with_pixels = round_2_repair_checked.join(round_2_repair_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left')
round_2_repair_results = round_2_repair_with_pixels.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repairs':a['repairs']+[r]},initial={'repairs':[]})
round_2_repair_planned = round_2_repair_requests.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repair_requests':a['repair_requests']+[{'parent_topic_index':r['parent_topic_index']}]},initial={'repair_requests':[]})
show(round_2_repair_planned)


### round_2_repair · ApplyTopicRepairs

In [ ]:
round_2_repair_assembled = await round_2_rows.join(round_2_repair_results, on='batch_id', how='left').join(round_2_repair_planned, on='batch_id', how='left').map(ApplyTopicRepairs()).checkpoint_async(tables / 'round_2_repair_assembled.jsonl', version=version)
show(round_2_repair_assembled)


### round_2_repair · RetainReviewedTopics

In [ ]:
round_2_repair_topics = await round_2_repair_assembled.flat_map(lambda r: r['rows']).map(RetainReviewedTopics()).checkpoint_async(tables / 'round_2_repair_topics.jsonl', version=version)
show(round_2_repair_topics, columns=['batch_id', 'topics'])


## 最后检查与保存

### residual · SelectRetainedParagraphs

In [ ]:
residual_retained = await round_2_repair_topics.map(SelectRetainedParagraphs()).checkpoint_async(tables / 'residual_retained.jsonl', version=version)
show(residual_retained, columns=['content'])


### residual · ParagraphRows

In [ ]:
residual_paragraphs = await residual_retained.map(lambda r: r['content']).flat_map(ParagraphRows()).checkpoint_async(tables / 'residual_paragraphs.jsonl', version=version)
show(residual_paragraphs)


### residual · EmbedParagraphBatch

In [ ]:
residual_embeddings = await residual_paragraphs.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/residual_embeddings', version=version).checkpoint_async(tables / 'residual_embeddings.jsonl', version=version)
show(residual_embeddings)


### residual · reduce_by_key

In [ ]:
residual_groups = await residual_embeddings.flat_map(lambda r: r['items']).reduce_by_key('concept', lambda a, r: {'concept': r['concept'], 'items': a['items'] + [r]}, initial={'items': []}).checkpoint_async(tables / 'residual_groups.jsonl', version=version)
show(residual_groups)


### residual · PlanCrossBatchReview

In [ ]:
residual_plans = await residual_groups.map(PlanCrossBatchReview(include_same_batch=True)).checkpoint_async(tables / 'residual_plans.jsonl', version=version)
show(residual_plans)


### residual · 展开待比较段落对

In [ ]:
residual_pairs = await residual_plans.flat_map(lambda r: r['review_requests']).map(lambda r: {**r, 'review_payload': {'items': r['items'], 'reasons': r['reasons']}}).checkpoint_async(tables / 'residual_pairs.jsonl', version=version)
show(residual_pairs, columns=['paragraph_ids', 'review_payload'])


### residual · review_cross_batch · 模型调用

In [ ]:
residual_responses = await residual_pairs.map_prompt_async('review_cross_batch', config='knowledge.yaml', inputs={'payload': 'review_payload'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'residual_responses.jsonl', version=version)
show(residual_responses, columns=['prompt_result', 'prompt_error'])


### residual · ApplyCrossBatchReview

In [ ]:
residual_reviews = await residual_responses.map_cached(ApplyCrossBatchReview(), cache_dir=RUN / 'cache/residual_relations', version=version).checkpoint_async(tables / 'residual_reviews.jsonl', version=version)
show(residual_reviews, columns=['paragraph_ids', 'relationship_review', 'next_action'])


### 保存完整段落及模型输入

In [ ]:
final_rows = await round_2_repair_topics.checkpoint_async(RUN / 'paragraphs.jsonl', version=version)
all_requests = joint_requests.union(initial_repair_requests).union(round_1_merge_requests).union(round_1_repair_requests).union(round_2_merge_requests).union(round_2_repair_requests)
await all_requests.checkpoint_async(RUN / 'requests.jsonl', version=version)
show(final_rows, columns=['batch_id', 'topics'])


### SourceCatalog · 整理来源链接

In [ ]:
catalogs = await related.map(SourceCatalog()).checkpoint_async(tables / 'catalogs.jsonl', version=version)
show(catalogs)


### SelectRetainedParagraphs → TopicRows · 展开最终主题

In [ ]:
topic_rows = await final_rows.map(SelectRetainedParagraphs()).map(lambda r: r['content']).flat_map(TopicRows()).checkpoint_async(tables / 'topic_rows.jsonl', version=version)
show(topic_rows)


### FormatTopicArticle · 标题、正文/图片、来源

In [ ]:
articles = await topic_rows.join(catalogs, on='concept', how='left').map(FormatTopicArticle()).checkpoint_async(tables / 'articles.jsonl', version=version)
show(articles, columns=['concept', 'article'])


### FinalKnowledgeRecord · 保存分层知识文件

In [ ]:
concept_articles = articles.reduce_by_key('concept', lambda a, r: {'concept': r['concept'], 'articles': a['articles'] + [r['article']]}, initial={'articles': []})
local_audit = round_2_merge_plan.map(lambda r: {'concept': r['concept'], 'local_audit': {'pending': r['pending'], 'local_task_count': len(r['requests'])}})
plan_audit = round_2_plans.map(lambda r: {'concept': r['concept'], 'cross_batch_plan': {'pending': r['pending'], 'candidate_count': len(r['review_requests']), 'passthrough_paragraph_ids': r['passthrough_paragraph_ids']}})
knowledge = await related.map(lambda r: {**r, 'concept': r['identity']['target_label']}).join(concept_articles, on='concept', how='left').join(local_audit, on='concept', how='left').join(plan_audit, on='concept', how='left').map(FinalKnowledgeRecord()).checkpoint_async(RUN / 'knowledge_base.jsonl', version=version)
show(knowledge, columns=['concept', 'knowledge'])
